

#1 Problem

We have:

$$
\epsilon u'' + (1+\epsilon) u' + u = 0, \quad u(0) = 0, \quad u(1) = 1,
$$
with $$(\epsilon = 0.01).$$



#2. Solve the ODE analytically

The equation is:

$$
0.01 \, u'' + 1.01 \, u' + u = 0.
$$

Characteristic equation:

$$
0.01 r^2 + 1.01 r + 1 = 0.
$$

Multiply by 100:

$$
r^2 + 101 r + 100 = 0.
$$

Discriminant:

$$
\Delta = 101^2 - 4 \cdot 100 = 10201 - 400 = 9801.
$$



So:

$$
r = \frac{-101 \pm 99}{2}.
$$

Roots:

$$
r_1 = \frac{-101 + 99}{2} = \frac{-2}{2} = -1,
$$
$$
r_2 = \frac{-101 - 99}{2} = \frac{-200}{2} = -100.
$$



General solution:

$$
u(x) = A e^{-x} + B e^{-100 x}.
$$



#3. Apply boundary conditions

$(u(0) = 0):$

$$
A + B = 0 \quad \Rightarrow \quad B = -A.$$

So:

$$
u(x) = A(e^{-x} - e^{-100 x}).
$$

$(u(1) = 1):$

$$
A(e^{-1} - e^{-100}) = 1.
$$

Since $(e^{-100} \approx 0)$ (about $(3.7\times 10^{-44})$), we can ignore $(e^{-100})$ for practical purposes.

Thus:

$$
A e^{-1} \approx 1 \quad \Rightarrow \quad A \approx e.
$$

More precisely:

$$
A = \frac{1}{e^{-1} - e^{-100}} = \frac{1}{e^{-1} - 0} \approx e.
$$

So \(A = e\) for all practical purposes.



Thus:

$$
u(x) \approx e(e^{-x} - e^{-100 x}) = e^{1-x} - e^{1 - 100 x}.
$$



In [5]:
import numpy as np
import matplotlib.pyplot as plt   ##FINITE DIFFERENCE SOLUTION

# Parameters
epsilon = 0.01


# 1. ANALYTICAL SOLUTION

def analytical_solution(x):
    """Exact analytical solution"""
    return np.exp(1 - x) - np.exp(1 - 100 * x)


# 2. FINITE DIFFERENCE SOLUTION

def finite_difference_solution(N=1000):
    """
    Solve using finite difference method
    epsilon*u'' + (1+epsilon)*u' + u = 0
    u(0)=0, u(1)=1
    """
    # Grid
    x = np.linspace(0, 1, N)
    h = x[1] - x[0]

    # Coefficients for the finite difference scheme
    # Central difference: u'' = (u_{i-1} - 2u_i + u_{i+1})/h^2
    # Central difference: u' = (u_{i+1} - u_{i-1})/(2h)

    # Build the matrix A and right-hand side b
    A = np.zeros((N, N))
    b = np.zeros(N)

    # Boundary conditions
    A[0, 0] = 1
    b[0] = 0  # u(0) = 0

    A[N-1, N-1] = 1
    b[N-1] = 1  # u(1) = 1

    # Interior points: epsilon*u'' + (1+epsilon)*u' + u = 0
    for i in range(1, N-1):
        # u'' coefficient
        A[i, i-1] += epsilon / (h**2)
        A[i, i]   += -2 * epsilon / (h**2)
        A[i, i+1] += epsilon / (h**2)

        # u' coefficient (central difference)
        A[i, i-1] += -(1 + epsilon) / (2 * h)
        A[i, i+1] += (1 + epsilon) / (2 * h)

        # u coefficient
        A[i, i] += 1

    # Solve the linear system
    u_numerical = np.linalg.solve(A, b)

    return x, u_numerical

# 3. VERIFICATION METHODS

def verify_ode_residual(u, x, epsilon):
    """Calculate residual of the ODE for verification"""
    h = x[1] - x[0]
    N = len(x)
    residual = np.zeros(N)

    # Central differences for interior points
    for i in range(1, N-1):
        u_xx = (u[i-1] - 2*u[i] + u[i+1]) / (h**2)
        u_x = (u[i+1] - u[i-1]) / (2 * h)
        residual[i] = epsilon * u_xx + (1 + epsilon) * u_x + u[i]

    return residual

def verify_boundary_conditions(u_analytical, u_numerical, x):
    """Verify boundary conditions are satisfied"""
    bc_left_analytical = u_analytical[0]
    bc_right_analytical = u_analytical[-1]

    bc_left_numerical = u_numerical[0]
    bc_right_numerical = u_numerical[-1]

    print("Boundary Condition Verification:")
    print(f"Analytical: u(0) = {bc_left_analytical:.10f}, u(1) = {bc_right_analytical:.10f}")
    print(f"Numerical:  u(0) = {bc_left_numerical:.10f}, u(1) = {bc_right_numerical:.10f}")
    print(f"BC errors: left = {abs(bc_left_numerical):.2e}, right = {abs(bc_right_numerical - 1):.2e}")

# 4. SOLVE AND COMPARE

# Generate solutions
x_fine = np.linspace(0, 1, 1000)
u_analytical = analytical_solution(x_fine)

x_num, u_numerical = finite_difference_solution(N=1000)

# Calculate residuals
residual_analytical = verify_ode_residual(u_analytical, x_fine, epsilon)
residual_numerical = verify_ode_residual(u_numerical, x_num, epsilon)

# 5. RESULTS


print("BOUNDARY VALUE PROBLEM SOLUTION VERIFICATION")

print(f"Equation: {epsilon}u'' + {1+epsilon}u' + u = 0")
print(f"BCs: u(0)=0, u(1)=1")
print()

# Verify boundary conditions
verify_boundary_conditions(u_analytical, u_numerical, x_fine)
print()

# Check maximum difference between solutions
u_analytical_on_num_grid = analytical_solution(x_num)
max_error = np.max(np.abs(u_numerical - u_analytical_on_num_grid))
print(f"Maximum difference between analytical and numerical: {max_error:.2e}")

# Check ODE residuals
print(f"\nODE Residual Analysis:")
print(f"Max analytical solution residual: {np.max(np.abs(residual_analytical)):.2e}")
print(f"Max numerical solution residual: {np.max(np.abs(residual_numerical)):.2e}")






print("ADDITIONAL VERIFICATION")


# Check if the solution satisfies the original characteristic equation
r1, r2 = -1, -100
print(f"Characteristic roots: r1 = {r1}, r2 = {r2}")

# Verify the fast mode is negligible away from boundary
x_test = 0.1
fast_mode = np.exp(1 - 100 * x_test)
slow_mode = np.exp(1 - x_test)
print(f"\nAt x = {x_test}:")
print(f"Slow mode: {slow_mode:.6f}")
print(f"Fast mode: {fast_mode:.6e}")
print(f"Fast/Slow ratio: {fast_mode/slow_mode:.2e}")

# Check derivative continuity
def derivative(x, u, h):
    """Calculate derivative using central differences"""
    du = np.zeros_like(u)
    du[1:-1] = (u[2:] - u[:-2]) / (2 * h)
    du[0] = (u[1] - u[0]) / h  # forward
    du[-1] = (u[-1] - u[-2]) / h  # backward
    return du

h_fine = x_fine[1] - x_fine[0]
du_analytical = derivative(x_fine, u_analytical, h_fine)

print(f"\nDerivative at x=0: {du_analytical[0]:.2f}")
print("This large derivative confirms the boundary layer at x=0")

BOUNDARY VALUE PROBLEM SOLUTION VERIFICATION
Equation: 0.01u'' + 1.01u' + u = 0
BCs: u(0)=0, u(1)=1

Boundary Condition Verification:
Analytical: u(0) = 0.0000000000, u(1) = 1.0000000000
Numerical:  u(0) = 0.0000000000, u(1) = 1.0000000000
BC errors: left = 6.82e-12, right = 0.00e+00

Maximum difference between analytical and numerical: 8.61e-04

ODE Residual Analysis:
Max analytical solution residual: 2.10e-01
Max numerical solution residual: 7.36e-12
ADDITIONAL VERIFICATION
Characteristic roots: r1 = -1, r2 = -100

At x = 0.1:
Slow mode: 2.459603
Fast mode: 1.234098e-04
Fast/Slow ratio: 5.02e-05

Derivative at x=0: 255.95
This large derivative confirms the boundary layer at x=0
